# Stage 2 — PAH (learned alpha) vs fixed-alpha, 5 obstacles — TWO-HEAD CRITIC + AUTO MULTI-SEED

**This version supersedes `stage2-pah-v0`.** Two changes vs v0:

1. **Two-head critic escalation** (`docs/research/01_pah_design.md` Section 9.2's
   closing note, triggered 2026-09-17): v0's weighted-prior fix
   (`prior_coef=0.15`) looked like a clean success on seed=42 (0/14 wrong
   direction), but across 5 seeds (42-46) it failed badly in 2 of them
   (seed-45: 65% wrong, seed-46: 89% wrong) — 34% wrong overall, always
   saturating at `alpha_max` near danger. Root cause: `A_mission` and
   `A_safety` shared ONE critic's values as their GAE baseline ("Option B
   lite"). That shared baseline can make `A_safety` unreliable enough, in
   some seeds, for the advantage-mixing actor loss to win its tug-of-war
   against the weighted prior. **Fix: two separate critic heads**
   (`Critic_mission`, `Critic_safety`), each trained on its own reward
   stream, each supplying its own GAE baseline. The weighted prior
   (`compute_prior_loss`, `prior_coef=0.15`) is UNCHANGED — this adds to
   it, does not replace it.

2. **Automatic multi-seed loop.** Instead of 5 separate hand-copied
   notebooks (one per seed, `stage2-pah-v0/var-alpha/seed-42` ...
   `seed-46`), this single notebook loops over `SEEDS = [42, 43, 44, 45, 46]`
   and trains each one in sequence, saving each seed's results to its own
   `output-seed{N}/` subfolder. Total runtime ≈ 5 × 44 min ≈ 3.7 hours,
   comfortably inside a single Kaggle GPU session.

**Config:** 5 drones · moving targets · 5 obstacles · conflict graph ON · warm-started
from Stage 2b's plain-MAPPO weights (`stage2-b/output/final_model.pt`) — **actor
only** now. Both critic heads start fresh every seed (there is no prior two-head
critic to load; Stage 2b's checkpoint has a single old-style `critic` key that
doesn't map onto `critic_m`/`critic_s`). PAH also always starts fresh, as before.

**One toggle decides the whole comparison** (`CFG['use_pah']` in Cell 5):

| `use_pah` | What runs | What it is |
|---|---|---|
| `True` | Alpha comes from the PAH network, trained with gradient | **M — the thesis method** (learned, state-dependent alpha) |
| `False` | Alpha is `CFG['fixed_alpha']`, a constant | **B4 — fixed-weight MAPPO baseline** (`docs/research/03_baseline_specs.md`) |

Run this notebook twice (once per `use_pah` value, in the two sibling files
`var-alpha/` and `fixed-alpha/`) — everything else identical. Both notebooks
use the SAME two-head-critic machinery: B4 also calls `compute_gae_two_head`
(alpha is just fixed at `CFG['fixed_alpha']` instead of coming from the PAH
network), so this fix and the multi-seed loop apply symmetrically to both.

**Before running on Kaggle:** upload `stage2-b/output/final_model.pt` as a Dataset
input (same as before) — Cell 6 checks a list of candidate paths and stops with a
clear error if it can't find it, before wasting any of the 5-seed loop's time.


In [ ]:
# Cell 1 — Imports
import torch
import numpy as np
from scipy.optimize import linear_sum_assignment
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device   : {DEVICE}')

In [ ]:
# Cell 2 — Conflict Graph
# Ported from code/algorithms/conflict_graph.py — unchanged from stage2-pah-v0.

import numpy as np

def _time_to_cpa(pos_i, pos_j, vel_i, vel_j):
    p = pos_i - pos_j
    v = vel_i - vel_j
    vv = np.dot(v, v)
    if vv < 1e-8:
        return np.inf
    return -np.dot(p, v) / vv

def _dist_at_cpa(pos_i, pos_j, vel_i, vel_j, horizon):
    p = pos_i - pos_j
    v = vel_i - vel_j
    vv = np.dot(v, v)
    if vv < 1e-8:
        return float(np.linalg.norm(p)), 0.0
    t_star = -np.dot(p, v) / vv
    t_clamped = float(np.clip(t_star, 0.0, horizon))
    return float(np.linalg.norm(p + v * t_clamped)), t_star

class ConflictGraph:
    K_NBR = 4

    def __init__(self, n_drones, horizon=3.0, d_danger=9.0):
        self.n      = n_drones
        self.H      = horizon
        self.d_dan  = d_danger
        self.adj    = np.zeros((n_drones, n_drones), dtype=bool)
        self.t_mat  = np.full((n_drones, n_drones), np.inf)

    def update(self, pos, vel):
        self.adj[:] = False
        self.t_mat[:] = np.inf
        for i in range(self.n):
            for j in range(i + 1, self.n):
                dcpa, t_star = _dist_at_cpa(pos[i], pos[j], vel[i], vel[j], self.H)
                self.t_mat[i, j] = self.t_mat[j, i] = t_star
                if dcpa < self.d_dan and 0.0 <= t_star <= self.H:
                    self.adj[i, j] = self.adj[j, i] = True

    def n_conflict(self, i):
        return int(self.adj[i].sum())

    def tau_collision(self, i):
        nbrs = np.where(self.adj[i])[0]
        if len(nbrs) == 0:
            return self.H
        return float(np.min(self.t_mat[i, nbrs]))

    def neighbor_obs(self, i, pos, vel):
        obs = np.zeros(self.K_NBR * 5, dtype=np.float32)
        nbrs = np.where(self.adj[i])[0]
        if len(nbrs) == 0:
            return obs
        order = np.argsort(self.t_mat[i, nbrs])
        nbrs  = nbrs[order]
        for slot, j in enumerate(nbrs[:self.K_NBR]):
            b = slot * 5
            obs[b:b+2]   = pos[j] - pos[i]
            obs[b+2:b+4] = vel[j] - vel[i]
            obs[b+4]     = 1.0
        return obs

In [ ]:
# Cell 3 — Environment (same as stage2-pah-v0, unchanged)
#
# step() returns info['pah_inputs'] = {tau, d_target, n_conflict} per drone,
# computed from the conflict graph and the current Hungarian assignment.

import numpy as np
from scipy.optimize import linear_sum_assignment

K_NBR    = 4
BASE_DIM = 10
OBS_DIM  = BASE_DIM + K_NBR * 5  # 30

class MultiUAVEnv:
    def __init__(self, n_drones=5, n_obstacles=5, world_size=500.0,
                 max_speed=5.0, max_steps=500, collision_radius=3.0,
                 target_radius=25.0, success_bonus=20.0,
                 target_speed=1.0, seed=None):
        self.n         = n_drones
        self.n_obs     = n_obstacles
        self.W         = world_size
        self.max_sp    = max_speed
        self.max_steps = max_steps
        self.col_r     = collision_radius
        self.tgt_r     = target_radius
        self.bonus     = success_bonus
        self.tgt_speed = target_speed
        self.rng       = np.random.default_rng(seed)
        self.cg        = ConflictGraph(n_drones, horizon=3.0,
                                       d_danger=collision_radius * 3.0)
        self.obs_dim   = OBS_DIM
        self.act_dim   = 2

    def reset(self):
        self.t                  = 0
        self.collision_occurred = False
        all_pos = self._sample_non_overlapping(self.n * 2, self.col_r * 2)
        self.drone_pos  = all_pos[:self.n].copy()
        self.target_pos = all_pos[self.n:].copy()
        self.obstacle_pos = self._place_obstacles(
            np.concatenate([self.drone_pos, self.target_pos]), self.n_obs
        )
        self.drone_vel = np.zeros((self.n, 2), dtype=np.float32)
        angles = self.rng.uniform(0, 2 * np.pi, self.n)
        self.target_vel = (self.tgt_speed * np.stack(
            [np.cos(angles), np.sin(angles)], axis=1
        )).astype(np.float32)
        self.assignment = self._hungarian()
        self.cg.update(self.drone_pos, self.drone_vel)
        return self._obs()

    def step(self, actions):
        self.t += 1
        actions = np.clip(actions, -self.max_sp, self.max_sp)
        self.drone_vel = actions.astype(np.float32)
        self.drone_pos = np.clip(self.drone_pos + self.drone_vel, 0.0, self.W)

        self.target_pos = self.target_pos + self.target_vel
        for k in range(self.n):
            for dim in range(2):
                if self.target_pos[k, dim] < 10.0:
                    self.target_pos[k, dim] = 10.0
                    self.target_vel[k, dim] *= -1.0
                elif self.target_pos[k, dim] > self.W - 10.0:
                    self.target_pos[k, dim] = self.W - 10.0
                    self.target_vel[k, dim] *= -1.0

        self.assignment = self._hungarian()
        self.cg.update(self.drone_pos, self.drone_vel)

        rewards, r_mission, r_safety = self._rewards()
        all_reached   = self._all_reached()
        any_collision = self._any_collision()
        timeout       = self.t >= self.max_steps

        if any_collision:
            self.collision_occurred = True

        terminated = all_reached or any_collision
        truncated  = timeout

        if all_reached and self.bonus:
            rewards   = rewards + self.bonus
            r_mission = r_mission + self.bonus   # bonus is a mission outcome, not safety

        pah_inputs = {
            'tau': np.array([self.cg.tau_collision(i) for i in range(self.n)], dtype=np.float32),
            'd_target': np.array([
                np.linalg.norm(self.drone_pos[i] - self.target_pos[self.assignment[i]])
                for i in range(self.n)
            ], dtype=np.float32),
            'n_conflict': np.array([self.cg.n_conflict(i) for i in range(self.n)], dtype=np.float32),
        }

        info = {
            'all_targets_reached': all_reached,
            'any_collision':       any_collision,
            'episode_collision':   self.collision_occurred,
            'r_mission':           r_mission,
            'r_safety':            r_safety,
            'pah_inputs':          pah_inputs,
        }
        return self._obs(), rewards, terminated, truncated, info

    def _obs(self):
        obs = np.zeros((self.n, self.obs_dim), dtype=np.float32)
        for i in range(self.n):
            rel = self.target_pos[self.assignment[i]] - self.drone_pos[i]
            cl  = self._clearances(i)
            base = np.concatenate([
                self.drone_pos[i] / self.W,
                self.drone_vel[i] / self.max_sp,
                rel               / self.W,
                cl,
            ])
            nbr = self.cg.neighbor_obs(i, self.drone_pos, self.drone_vel)
            for s in range(K_NBR):
                nbr[s*5 : s*5+2]   /= self.W
                nbr[s*5+2 : s*5+4] /= self.max_sp
            obs[i] = np.concatenate([base, nbr])
        return obs

    def _rewards(self):
        r_mission = np.zeros(self.n, dtype=np.float32)
        r_safety  = np.zeros(self.n, dtype=np.float32)
        d_danger  = self.col_r * 3.0
        zone_w    = d_danger - self.col_r
        for i in range(self.n):
            rel  = self.target_pos[self.assignment[i]] - self.drone_pos[i]
            dist = np.linalg.norm(rel)
            direction  = rel / (dist + 1e-6)
            r_progress = float(np.dot(self.drone_vel[i], direction) / self.max_sp)
            r_mission[i] = 0.4 * r_progress + 0.3 * (-dist / self.W)
            min_d = float('inf')
            for j in range(self.n):
                if j != i:
                    min_d = min(min_d, np.linalg.norm(self.drone_pos[i] - self.drone_pos[j]))
            for op in self.obstacle_pos:
                min_d = min(min_d, np.linalg.norm(self.drone_pos[i] - op))
            if min_d < self.col_r:
                r_safety[i] = -1.0
            elif min_d < d_danger:
                r_safety[i] = -((d_danger - min_d) / zone_w)
        rewards = r_mission + 0.3 * r_safety
        return rewards, r_mission, r_safety

    def _hungarian(self):
        cost = np.array([
            [np.linalg.norm(self.drone_pos[i] - self.target_pos[j])
             for j in range(self.n)]
            for i in range(self.n)
        ])
        _, col = linear_sum_assignment(cost)
        return col

    def _all_reached(self):
        return all(
            np.linalg.norm(self.drone_pos[i] - self.target_pos[self.assignment[i]]) <= self.tgt_r
            for i in range(self.n)
        )

    def _any_collision(self):
        for i in range(self.n):
            for j in range(self.n):
                if j != i and np.linalg.norm(self.drone_pos[i] - self.drone_pos[j]) < self.col_r:
                    return True
            for op in self.obstacle_pos:
                if np.linalg.norm(self.drone_pos[i] - op) < self.col_r:
                    return True
        return False

    def _clearances(self, i):
        pos = self.drone_pos[i]
        cl  = np.array([self.W - pos[1], pos[1], self.W - pos[0], pos[0]], dtype=np.float32)
        for op in self.obstacle_pos:
            diff = op - pos
            if diff[1] > 0: cl[0] = min(cl[0],  diff[1])
            else:           cl[1] = min(cl[1], -diff[1])
            if diff[0] > 0: cl[2] = min(cl[2],  diff[0])
            else:           cl[3] = min(cl[3], -diff[0])
        return cl / self.W

    def _sample_non_overlapping(self, n, min_dist):
        pts, attempts = [], 0
        while len(pts) < n:
            attempts += 1
            if attempts > 10000:
                raise RuntimeError('World too crowded.')
            c = self.rng.uniform(10.0, self.W - 10.0, 2)
            if not any(np.linalg.norm(c - p) < min_dist for p in pts):
                pts.append(c)
        return np.array(pts, dtype=np.float32)

    def _place_obstacles(self, existing, n_obs, max_tries=1000):
        if n_obs <= 0:
            return np.zeros((0, 2), dtype=np.float32)
        r_min = 2.0 * self.col_r
        placed = []
        for _ in range(n_obs):
            for _ in range(max_tries):
                c = self.rng.uniform(10.0, self.W - 10.0, 2)
                if any(np.linalg.norm(c - o) < r_min for o in placed): continue
                if any(np.linalg.norm(c - p) < self.col_r for p in existing): continue
                placed.append(c)
                break
            else:
                break
        return np.array(placed, dtype=np.float32) if placed else np.zeros((0, 2), dtype=np.float32)

print('obs_dim = {}  (10 base + {}x5 conflict neighbors, normalized)'.format(OBS_DIM, K_NBR))
print('step() returns info["pah_inputs"] = {tau, d_target, n_conflict}')


In [ ]:
# Cell 4 — MAPPO + PAH (Option B, TWO-HEAD CRITIC) — Actor, Critic x2, PAH, Buffer, Agent
#
# TWO-HEAD CRITIC ESCALATION (2026-09-17, docs/research/01_pah_design.md
# Section 9.2's closing note, triggered after the weighted-prior fix showed
# 34% wrong-direction rate across 5 seeds — see Cell 0 markdown and
# sessions/2026-09-17.md Sections 13-14 for the full diagnosis).
#
# BEFORE (stage2-pah-v0): ONE shared critic V(s). A_mission and A_safety both
# used V(s)'s values as their GAE baseline ("Option B lite"). That shared
# baseline could make A_safety read as unreliable enough, in some seeds, for
# the advantage-mixing actor loss below to push alpha the WRONG way near
# danger even with the weighted prior fighting it.
#
# NOW: TWO separate critic heads, Critic_mission and Critic_safety, each
# trained on its own reward stream (r_mission, r_safety respectively), each
# supplying its own GAE baseline. This removes the shared-baseline path for
# A_safety to be systematically wrong. The weighted prior from v0
# (compute_prior_loss, prior_coef=0.15) is UNCHANGED — this is additive, not
# a replacement.
#
# ONE toggle, CFG['use_pah']:
#   True  -> alpha comes from the PAH network, trained with gradient (M — the
#            thesis method)
#   False -> alpha is CFG['fixed_alpha'], a constant, no PAH network at all
#            (B4 baseline — docs/research/03_baseline_specs.md). B4 ALSO uses
#            the two-head critic (same GAE machinery), only where alpha comes
#            from differs — keeps the comparison to a single isolated variable.

import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


class Actor(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden),  nn.Tanh(),
        )
        self.mu_head = nn.Linear(hidden, act_dim)
        self.log_std = nn.Parameter(torch.zeros(act_dim))

    def forward(self, x):
        h       = self.net(x)
        mu      = self.mu_head(h)
        log_std = self.log_std.clamp(-4.0, 1.5)   # safety net, see Stage 2 notes
        std     = log_std.exp().expand_as(mu)
        return Normal(mu, std)

    def get_action(self, obs):
        dist   = self(obs)
        action = dist.sample()
        log_p  = dist.log_prob(action).sum(-1)
        return action, log_p

    def evaluate_action(self, obs, action):
        dist    = self(obs)
        log_p   = dist.log_prob(action).sum(-1)
        entropy = dist.entropy().sum(-1)
        return log_p, entropy


class Critic(nn.Module):
    """One value head — instantiated TWICE now (critic_m, critic_s) instead of
    once for a combined reward. Architecture itself is unchanged."""
    def __init__(self, global_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(global_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden),     nn.Tanh(),
            nn.Linear(hidden, 1),
        )

    def forward(self, global_state):
        return self.net(global_state).squeeze(-1)


class PAHNormalizer:
    """Scales the 3 raw PAH inputs to roughly [0,1]. Unchanged from stage2-pah-v0."""
    def __init__(self, horizon=3.0, d_max=707.1, n_drones=5):
        self.horizon  = horizon
        self.d_max    = d_max
        self.n_drones = n_drones

    def normalize(self, tau, d_target, n_conflict):
        tau_norm = tau.clamp(0.0, self.horizon) / self.horizon
        d_norm   = d_target.clamp(0.0, self.d_max) / self.d_max
        denom    = max(self.n_drones - 1, 1)
        n_norm   = n_conflict.clamp(0.0, denom) / denom
        return torch.stack([tau_norm, d_norm, n_norm], dim=-1)


class PriorityArbitrationHead(nn.Module):
    """Maps normalized (tau, d_target, n_conflict) -> alpha in [alpha_min, alpha_max].
    UNCHANGED from stage2-pah-v0 (weighted prior fix, verified working seed-42-44,
    partially failing 45-46 — this is what the two-head critic is meant to help)."""
    def __init__(self, hidden_dim=32, alpha_min=0.1, alpha_max=0.9, prior_coef=0.15):
        super().__init__()
        self.alpha_min  = alpha_min
        self.alpha_max  = alpha_max
        self.prior_coef = prior_coef
        self.net = nn.Sequential(
            nn.Linear(3, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )
        nn.init.zeros_(self.net[-1].bias)
        nn.init.xavier_uniform_(self.net[-1].weight, gain=0.1)

    def forward(self, x):
        raw   = self.net(x)
        alpha = torch.sigmoid(raw)
        return self.alpha_min + (self.alpha_max - self.alpha_min) * alpha

    def compute_prior_loss(self, alpha, tau_norm):
        """Pulls alpha toward alpha_min near danger, alpha_max when safe, WEIGHTED
        so danger-close samples dominate the mean. Unchanged from stage2-pah-v0."""
        if self.prior_coef == 0.0:
            return torch.tensor(0.0, device=alpha.device)
        alpha_target = self.alpha_min + (self.alpha_max - self.alpha_min) * tau_norm
        weight = 0.1 + 0.9 * (1.0 - tau_norm)   # 1.0 at danger, 0.1 when safe
        return self.prior_coef * (weight * (alpha.squeeze(-1) - alpha_target).pow(2)).mean()


class RolloutBuffer:
    """Per-drone rewards/log-probs/PAH-inputs kept separate throughout. Now also
    keeps TWO value streams (vals_m, vals_s) instead of one combined 'vals' —
    each critic head's own estimate, needed for its own GAE baseline."""
    def __init__(self, n_drones):
        self.n_drones = n_drones
        self.clear()

    def clear(self):
        self.obs, self.acts = [], []
        self.vals_m, self.vals_s, self.lps, self.dones = [], [], [], []
        self.r_mission, self.r_safety = [], []
        self.tau, self.d_target, self.n_conflict = [], [], []

    def add(self, obs, act, val_m, val_s, lp, done, r_mission, r_safety, pah_inputs):
        self.obs.append(obs.copy())
        self.acts.append(act.copy())
        self.vals_m.append(float(val_m))
        self.vals_s.append(float(val_s))
        self.lps.append(lp.copy())
        self.dones.append(bool(done))
        self.r_mission.append(r_mission.copy())
        self.r_safety.append(r_safety.copy())
        self.tau.append(pah_inputs['tau'].copy())
        self.d_target.append(pah_inputs['d_target'].copy())
        self.n_conflict.append(pah_inputs['n_conflict'].copy())

    def __len__(self):
        return len(self.r_mission)

    def compute_gae_two_head(self, last_val_m, last_val_s, gamma, lam):
        """Separate GAE for r_mission (baseline = critic_m's own values) and
        r_safety (baseline = critic_s's own values) — the two-head fix. Each
        stream is now fully independent, no shared baseline."""
        T = len(self.r_mission)
        n = self.n_drones
        rm     = np.array(self.r_mission, dtype=np.float32)
        rs     = np.array(self.r_safety,  dtype=np.float32)
        vm     = np.array(self.vals_m,    dtype=np.float32)
        vs     = np.array(self.vals_s,    dtype=np.float32)
        dones  = np.array(self.dones,     dtype=np.float32)
        adv_m = np.zeros((T, n), dtype=np.float32)
        adv_s = np.zeros((T, n), dtype=np.float32)
        gae_m = np.zeros(n, dtype=np.float32)
        gae_s = np.zeros(n, dtype=np.float32)
        for t in reversed(range(T)):
            next_vm = last_val_m if t == T - 1 else vm[t + 1]
            next_vs = last_val_s if t == T - 1 else vs[t + 1]
            mask    = 1.0 - dones[t]
            delta_m = rm[t] + gamma * next_vm * mask - vm[t]
            delta_s = rs[t] + gamma * next_vs * mask - vs[t]
            gae_m   = delta_m + gamma * lam * mask * gae_m
            gae_s   = delta_s + gamma * lam * mask * gae_s
            adv_m[t] = gae_m
            adv_s[t] = gae_s
        ret_m = adv_m + vm[:, np.newaxis]
        ret_s = adv_s + vs[:, np.newaxis]
        return adv_m, adv_s, ret_m, ret_s


class MAPPOAgent:
    def __init__(self, n_drones, obs_dim, act_dim, use_pah=True, fixed_alpha=0.5,
                 pah_hidden=32, alpha_min=0.1, alpha_max=0.9, pah_prior_coef=0.15,
                 pah_horizon=3.0, pah_d_max=707.1,
                 lr=3e-4, gamma=0.99, lam=0.95, clip_eps=0.2, vf_coef=0.5,
                 ent_coef=0.003, n_epochs=10, batch_size=64):
        self.n_drones = n_drones
        self.use_pah  = use_pah
        self.fixed_alpha = fixed_alpha
        self.gamma, self.lam           = gamma, lam
        self.clip_eps                  = clip_eps
        self.vf_coef, self.ent_coef    = vf_coef, ent_coef
        self.n_epochs, self.batch_size = n_epochs, batch_size

        global_dim   = n_drones * obs_dim
        self.actor   = Actor(obs_dim, act_dim).to(DEVICE)
        self.critic_m = Critic(global_dim).to(DEVICE)   # baseline for A_mission
        self.critic_s = Critic(global_dim).to(DEVICE)   # baseline for A_safety
        params = (list(self.actor.parameters()) + list(self.critic_m.parameters())
                  + list(self.critic_s.parameters()))

        if use_pah:
            self.pah        = PriorityArbitrationHead(
                pah_hidden, alpha_min, alpha_max, pah_prior_coef
            ).to(DEVICE)
            self.normalizer = PAHNormalizer(pah_horizon, pah_d_max, n_drones)
            params += list(self.pah.parameters())
        else:
            self.pah        = None
            self.normalizer = None

        self.opt = optim.Adam(params, lr=lr)

    @torch.no_grad()
    def get_actions(self, obs):
        obs_t          = torch.tensor(obs, dtype=torch.float32, device=DEVICE)
        action_t, lp_t = self.actor.get_action(obs_t)
        global_state   = obs_t.flatten().unsqueeze(0)
        val_m = self.critic_m(global_state).item()
        val_s = self.critic_s(global_state).item()
        return action_t.cpu().numpy(), lp_t.cpu().numpy(), val_m, val_s

    @torch.no_grad()
    def compute_alpha(self, tau_np, d_np, n_np):
        """Inference-time alpha (no gradient)."""
        if not self.use_pah:
            return np.full(self.n_drones, self.fixed_alpha, dtype=np.float32)
        tau = torch.tensor(tau_np, dtype=torch.float32, device=DEVICE)
        d   = torch.tensor(d_np,   dtype=torch.float32, device=DEVICE)
        n   = torch.tensor(n_np,   dtype=torch.float32, device=DEVICE)
        x     = self.normalizer.normalize(tau, d, n)
        alpha = self.pah(x).squeeze(-1)
        return alpha.cpu().numpy()

    def _alpha_with_grad(self, tau_t, d_t, n_t):
        """Training-time alpha (with gradient) for the actor loss.
        Returns (alpha, tau_norm)."""
        if not self.use_pah:
            alpha = torch.full((tau_t.shape[0],), self.fixed_alpha,
                                dtype=torch.float32, device=DEVICE)
            return alpha, None
        x = self.normalizer.normalize(tau_t, d_t, n_t)
        return self.pah(x).squeeze(-1), x[..., 0]

    def update(self, buffer, last_obs):
        last_obs_t = torch.tensor(last_obs, dtype=torch.float32, device=DEVICE)
        with torch.no_grad():
            last_val_m = self.critic_m(last_obs_t.flatten().unsqueeze(0)).item()
            last_val_s = self.critic_s(last_obs_t.flatten().unsqueeze(0)).item()

        adv_m_np, adv_s_np, ret_m_np, ret_s_np = buffer.compute_gae_two_head(
            last_val_m, last_val_s, self.gamma, self.lam
        )

        obs_np = np.array(buffer.obs,  dtype=np.float32)   # (T,n,D)
        act_np = np.array(buffer.acts, dtype=np.float32)   # (T,n,2)
        lp_np  = np.array(buffer.lps,  dtype=np.float32)   # (T,n)
        tau_np = np.array(buffer.tau,        dtype=np.float32)  # (T,n)
        d_np   = np.array(buffer.d_target,   dtype=np.float32)  # (T,n)
        nc_np  = np.array(buffer.n_conflict, dtype=np.float32)  # (T,n)
        T, N, D = obs_np.shape

        adv_m_t = torch.tensor(adv_m_np, dtype=torch.float32, device=DEVICE).reshape(-1)
        adv_s_t = torch.tensor(adv_s_np, dtype=torch.float32, device=DEVICE).reshape(-1)
        adv_m_flat = (adv_m_t - adv_m_t.mean()) / (adv_m_t.std() + 1e-8)
        adv_s_flat = (adv_s_t - adv_s_t.mean()) / (adv_s_t.std() + 1e-8)

        ret_m_t = torch.tensor(ret_m_np, dtype=torch.float32, device=DEVICE).reshape(-1)
        ret_s_t = torch.tensor(ret_s_np, dtype=torch.float32, device=DEVICE).reshape(-1)

        obs_flat = torch.tensor(obs_np.reshape(T * N, D), dtype=torch.float32, device=DEVICE)
        act_flat = torch.tensor(act_np.reshape(T * N, -1), dtype=torch.float32, device=DEVICE)
        lp_flat  = torch.tensor(lp_np.reshape(T * N),      dtype=torch.float32, device=DEVICE)
        tau_flat = torch.tensor(tau_np.reshape(T * N), dtype=torch.float32, device=DEVICE)
        d_flat   = torch.tensor(d_np.reshape(T * N),   dtype=torch.float32, device=DEVICE)
        nc_flat  = torch.tensor(nc_np.reshape(T * N),  dtype=torch.float32, device=DEVICE)

        gs_t   = obs_flat.reshape(T, N * D)
        gs_exp = gs_t.unsqueeze(1).expand(T, N, -1).reshape(T * N, -1)

        idx_all = np.arange(T * N)
        a_losses, v_losses, entropies, pah_losses, alphas_seen = [], [], [], [], []

        for _ in range(self.n_epochs):
            np.random.shuffle(idx_all)
            for start in range(0, T * N, self.batch_size):
                idx = idx_all[start:start + self.batch_size]

                log_p, entropy = self.actor.evaluate_action(obs_flat[idx], act_flat[idx])

                alpha, tau_norm = self._alpha_with_grad(tau_flat[idx], d_flat[idx], nc_flat[idx])
                w_adv = alpha * adv_m_flat[idx] + (1.0 - alpha) * adv_s_flat[idx]

                ratio  = (log_p - lp_flat[idx]).exp()
                surr1  = ratio * w_adv
                surr2  = ratio.clamp(1 - self.clip_eps, 1 + self.clip_eps) * w_adv
                a_loss = -torch.min(surr1, surr2).mean()

                v_pred_m = self.critic_m(gs_exp[idx])
                v_pred_s = self.critic_s(gs_exp[idx])
                v_loss_m = 0.5 * (v_pred_m - ret_m_t[idx]).pow(2).mean()
                v_loss_s = 0.5 * (v_pred_s - ret_s_t[idx]).pow(2).mean()
                v_loss   = v_loss_m + v_loss_s

                e_loss = -entropy.mean()
                loss = a_loss + self.vf_coef * v_loss + self.ent_coef * e_loss

                if self.use_pah:
                    pah_prior = self.pah.compute_prior_loss(alpha, tau_norm)
                    loss = loss + pah_prior
                    pah_losses.append(pah_prior.item())

                self.opt.zero_grad()
                loss.backward()
                params = (list(self.actor.parameters()) + list(self.critic_m.parameters())
                          + list(self.critic_s.parameters()))
                if self.use_pah:
                    params += list(self.pah.parameters())
                nn.utils.clip_grad_norm_(params, 0.5)
                self.opt.step()

                a_losses.append(a_loss.item())
                v_losses.append(v_loss.item())
                entropies.append(-e_loss.item())
                alphas_seen.append(alpha.detach().mean().item())

        stats = {
            'actor_loss':  float(np.mean(a_losses)),
            'critic_loss': float(np.mean(v_losses)),
            'entropy':     float(np.mean(entropies)),
            'alpha_mean':  float(np.mean(alphas_seen)),
            'alpha_std':   float(np.std(alphas_seen)),
        }
        if pah_losses:
            stats['pah_loss'] = float(np.mean(pah_losses))
        return stats

    def save(self, path):
        ckpt = {
            'actor':    self.actor.state_dict(),
            'critic_m': self.critic_m.state_dict(),
            'critic_s': self.critic_s.state_dict(),
        }
        if self.use_pah:
            ckpt['pah'] = self.pah.state_dict()
        torch.save(ckpt, path)

    def load_actor_only(self, path):
        """Warm-start: load ONLY the actor. Both critic heads always start
        fresh here. SUPERSEDED as the default (2026-09-18) -- see
        load_actor_and_dup_critic() below; fresh-random critics paired with
        an already-converged actor caused alpha to lock into a uniformly
        WRONG direction near danger (11/11 wrong on seed 42, worse than the
        single-critic version's 0/14) -- see sessions/2026-09-18.md. Kept
        here as a documented ablation point."""
        ckpt = torch.load(path, map_location=DEVICE, weights_only=True)
        self.actor.load_state_dict(ckpt['actor'])

    def load_actor_and_dup_critic(self, path):
        """Warm-start: load the actor AND duplicate an old single-critic
        checkpoint's weights into BOTH new critic heads (critic_m, critic_s).
        Added 2026-09-18 to fix load_actor_only's fresh-critic problem (see
        its docstring above and sessions/2026-09-18.md) -- gives both heads
        a reasonable starting point instead of random init, while they still
        diverge during training since each trains on its own reward stream."""
        ckpt = torch.load(path, map_location=DEVICE, weights_only=True)
        self.actor.load_state_dict(ckpt['actor'])
        if 'critic' in ckpt:
            self.critic_m.load_state_dict(ckpt['critic'])
            self.critic_s.load_state_dict(ckpt['critic'])
        else:
            self.critic_m.load_state_dict(ckpt['critic_m'])
            self.critic_s.load_state_dict(ckpt['critic_s'])


print('MAPPO + PAH (Option B, two-head critic) loaded.')
print('use_pah toggles learned-alpha vs fixed-alpha (baseline B4) through the exact same code path.')
print('Two-head critic: A_mission and A_safety now use independent value baselines (critic_m, critic_s).')


In [ ]:
# Cell 5 — Config (two-head critic, auto multi-seed)
#
# SEEDS list drives Cell 6's loop — trains each seed in sequence, one Kaggle
# session, saving each to its own output-seed{N}/ subfolder. No more
# hand-copied per-seed notebooks (stage2-pah-v0's approach).

import os

CFG = {
    # Environment — identical to Stage 2b / stage2-pah-v0
    'n_drones':         5,
    'n_obstacles':      5,
    'world_size':       500.0,
    'max_speed':        5.0,
    'max_steps':        500,
    'target_radius':    25.0,
    'collision_radius': 3.0,
    'success_bonus':    20.0,
    'target_speed':     1.0,
    # Training — identical to Stage 2b / stage2-pah-v0
    'total_episodes':   8000,
    'rollout_steps':    512,
    'eval_every':       100,
    'eval_episodes':    20,
    'save_every':       500,
    # MAPPO — identical to Stage 2b / stage2-pah-v0
    'lr':               3e-4,
    'gamma':            0.99,
    'lam':              0.95,
    'clip_eps':         0.2,
    'vf_coef':          0.5,
    'ent_coef':         0.003,
    'n_epochs':         10,
    'batch_size':       64,
    # --- THE EXPERIMENT TOGGLE -------------------------------------------
    'use_pah':          False,  # B4 -- fixed alpha baseline
    'fixed_alpha':      0.5,    # only used when use_pah=False — try 0.3 / 0.5 / 0.7
    # PAH hyperparameters (only used when use_pah=True) — UNCHANGED from
    # stage2-pah-v0's weighted-prior fix (verified working seeds 42-44,
    # partially failing 45-46 — see Cell 0 markdown)
    'pah_hidden_dim':   32,
    'alpha_min':        0.1,
    'alpha_max':        0.9,
    'pah_prior_coef':   0.15,
    'pah_horizon':      3.0,                 # matches ConflictGraph horizon in Cell 2/3
    'pah_d_max':        500.0 * 1.4142136,   # world diagonal
}

# --- MULTI-SEED DRIVER ---------------------------------------------------
# TEMPORARY (2026-09-18): only seed 42 for now, to sanity-check the
# load_actor_and_dup_critic warm-start fix (~44 min) before committing to
# the full 3.7h / 5-seed run. Once seed 42 looks right (danger-close alpha
# no longer uniformly wrong), change this back to [42, 43, 44, 45, 46].
SEEDS = [42]

RUN_TAG = 'M-learned-alpha' if CFG['use_pah'] else 'B4-fixed-alpha-{:.1f}'.format(CFG['fixed_alpha'])

# Warm-start: Stage 2b's plain-MAPPO weights. As of 2026-09-18, loads the
# actor AND duplicates the old single critic into BOTH new heads (critic_m,
# critic_s) via load_actor_and_dup_critic() -- see that method's docstring
# in Cell 4 for why (load_actor_only's fresh-random critics caused alpha to
# lock into a uniformly wrong direction near danger). Candidate paths checked in order.
WARM_START_CANDIDATES = [
    '../../stage2-b/output/final_model.pt',                  # local run, relative to this notebook
    '/kaggle/input/stage2b-checkpoint/final_model.pt',        # Kaggle: dataset named "stage2b-checkpoint"
    '/kaggle/input/stage2b-output/final_model.pt',            # Kaggle: alternate dataset name
]
_warm_path = None
for _candidate in WARM_START_CANDIDATES:
    if os.path.exists(_candidate):
        _warm_path = _candidate
        break
if _warm_path is None:
    raise FileNotFoundError(
        "This notebook needs Stage 2b's final_model.pt to warm-start from, but none "
        "of these paths exist:\n  " + "\n  ".join(WARM_START_CANDIDATES) +
        "\n\nOn Kaggle: attach stage2-b/output/final_model.pt as a Dataset input "
        "and add its path to WARM_START_CANDIDATES above.\n"
        "Checking this BEFORE the 5-seed loop starts — no point burning Kaggle time "
        "on seed 42 only to fail on a missing file."
    )

print('Config loaded.')
print('  Mode     = {}'.format('M (learned alpha, PAH ON)' if CFG['use_pah']
                               else 'B4 (fixed alpha = {})'.format(CFG['fixed_alpha'])))
print('  Seeds    = {}'.format(SEEDS))
print('  RUN_TAG  = {}'.format(RUN_TAG))
print('  Warm-start actor from: {}'.format(_warm_path))


In [ ]:
# Cell 6 — Train ALL SEEDS in sequence (two-head critic)
#
# Reuses the fixed Stage 2 training-loop pattern (per-drone rewards/log-probs,
# no averaging, last_eval/last_save so the modulo-skip bug can't recur — see
# sessions/2026-09-16.md Parts 12-13). NEW here vs stage2-pah-v0:
#   - get_actions() now returns (val_m, val_s) instead of one value
#   - buffer.add() stores both, compute_gae_two_head() uses both independently
#   - the whole per-seed run is wrapped in run_training(seed) and called once
#     per entry in SEEDS (Cell 5) -- no more separate hand-copied notebooks

import time, json as json_module

def make_env(seed, bonus=CFG['success_bonus']):
    return MultiUAVEnv(
        n_drones         = CFG['n_drones'],
        n_obstacles      = CFG['n_obstacles'],
        world_size       = CFG['world_size'],
        max_speed        = CFG['max_speed'],
        max_steps        = CFG['max_steps'],
        collision_radius = CFG['collision_radius'],
        target_radius    = CFG['target_radius'],
        success_bonus    = bonus,
        target_speed     = CFG['target_speed'],
        seed             = seed,
    )

# --- DIAGNOSTIC: obs range check before the whole loop starts ---
_diag_env = make_env(seed=SEEDS[0])
_diag_obs = _diag_env.reset()
print("[DIAG] obs min={:.4f}  max={:.4f}  shape={}".format(
    _diag_obs.min(), _diag_obs.max(), _diag_obs.shape))
assert _diag_obs.max() < 10.0, "STOP: obs unnormalized!"
print("[DIAG] obs range OK -- starting {}-seed loop\n".format(len(SEEDS)))
# -----------------------------------------------------------------

import matplotlib.pyplot as plt

def plot_results(history, results_dir, seed):
    """Per-seed results plot -- defined here (not a later cell) because
    run_training() below calls it immediately after each seed finishes, so
    the whole loop's progress is visible without waiting for all 5 seeds."""
    n_panels = 4 if CFG['use_pah'] else 3
    fig, axes = plt.subplots(1, n_panels, figsize=(5 * n_panels, 4))
    mode_str = 'M -- learned alpha (PAH ON)' if CFG['use_pah'] else 'B4 -- fixed alpha = {}'.format(CFG['fixed_alpha'])
    fig.suptitle('Stage 2 (two-head critic) -- seed {} -- {}'.format(seed, mode_str), fontweight='bold')

    axes[0].plot(history['episode'], [s*100 for s in history['success']], color='green', linewidth=2, label='Success')
    axes[0].plot(history['episode'], [c*100 for c in history['collision']], 'r--', linewidth=1.5, label='Collision')
    axes[0].set_title('Success vs Collision Rate')
    axes[0].set_ylabel('%'); axes[0].set_xlabel('Episode')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(history['episode'], history['actor_loss'],  color='blue',   label='Actor Loss')
    axes[1].plot(history['episode'], history['critic_loss'], color='orange', label='Critic Loss (m+s)')
    axes[1].set_title('Training Losses')
    axes[1].set_xlabel('Episode')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    axes[2].plot(history['episode'], history['entropy'], color='purple', linewidth=2)
    axes[2].axhline(0, color='gray', linewidth=0.8, linestyle=':')
    axes[2].set_title('Policy Entropy')
    axes[2].set_xlabel('Episode')
    axes[2].grid(True, alpha=0.3)

    if CFG['use_pah']:
        axes[3].plot(history['episode'], history['alpha_mean'], color='teal', linewidth=2, label='alpha (mean)')
        axes[3].fill_between(
            history['episode'],
            [m - s for m, s in zip(history['alpha_mean'], history['alpha_std'])],
            [m + s for m, s in zip(history['alpha_mean'], history['alpha_std'])],
            color='teal', alpha=0.15,
        )
        axes[3].plot(history['episode'], history['eval_alpha_low_tau'],  'r:', linewidth=1.5, label='alpha | danger close')
        axes[3].plot(history['episode'], history['eval_alpha_high_tau'], 'b:', linewidth=1.5, label='alpha | safe')
        axes[3].axhline(CFG['alpha_min'], color='gray', linewidth=0.6, linestyle='--')
        axes[3].axhline(CFG['alpha_max'], color='gray', linewidth=0.6, linestyle='--')
        axes[3].set_ylim(0, 1)
        axes[3].set_title('Alpha -- collapse/direction check')
        axes[3].set_xlabel('Episode')
        axes[3].legend(fontsize=8); axes[3].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('{}/training_curves.png'.format(results_dir), dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print('Plot saved for seed {}.'.format(seed))


def run_training(seed):
    print("\n" + "=" * 80)
    print("SEED {} -- {}".format(seed, RUN_TAG))
    print("=" * 80)

    np.random.seed(seed)
    torch.manual_seed(seed)

    run_name    = '{}-seed{}'.format(RUN_TAG, seed)
    RESULTS_DIR = os.path.join(os.getcwd(), 'output-seed{}'.format(seed))
    os.makedirs(RESULTS_DIR, exist_ok=True)

    env      = make_env(seed=seed)
    eval_env = make_env(seed=seed, bonus=0.0)
    agent    = MAPPOAgent(
        n_drones        = CFG['n_drones'],
        obs_dim         = OBS_DIM,
        act_dim         = 2,
        use_pah         = CFG['use_pah'],
        fixed_alpha     = CFG['fixed_alpha'],
        pah_hidden      = CFG['pah_hidden_dim'],
        alpha_min       = CFG['alpha_min'],
        alpha_max       = CFG['alpha_max'],
        pah_prior_coef  = CFG['pah_prior_coef'],
        pah_horizon     = CFG['pah_horizon'],
        pah_d_max       = CFG['pah_d_max'],
        lr              = CFG['lr'],
        gamma           = CFG['gamma'],
        lam             = CFG['lam'],
        clip_eps        = CFG['clip_eps'],
        vf_coef         = CFG['vf_coef'],
        ent_coef        = CFG['ent_coef'],
        n_epochs        = CFG['n_epochs'],
        batch_size      = CFG['batch_size'],
    )

    agent.load_actor_and_dup_critic(_warm_path)
    print("Warm-started actor + both critic heads (duplicated) from: {}".format(_warm_path))
    print("(critic_m and critic_s start as copies of the old single critic, then diverge; PAH -- if enabled -- always starts fresh)")

    buffer   = RolloutBuffer(n_drones=CFG['n_drones'])
    history  = {
        'episode': [], 'success': [], 'collision': [],
        'actor_loss': [], 'critic_loss': [], 'entropy': [], 'log_std': [],
        'alpha_mean': [], 'alpha_std': [], 'pah_loss': [],
        'eval_alpha_low_tau': [], 'eval_alpha_high_tau': [],
    }
    obs       = env.reset()
    ep_count  = 0
    last_eval = 0
    last_save = 0
    start     = time.time()

    header = " {:>9} | {:>8} | {:>10} | {:>8} | {:>8} | {:>10} | {:>7}".format(
        "Episode", "Success", "Collision", "A-Loss", "Entropy", "alpha(mu)", "Time"
    )
    print(header)
    print('-' * 75)

    while ep_count < CFG['total_episodes']:
        for _ in range(CFG['rollout_steps']):
            acts, lps, val_m, val_s = agent.get_actions(obs)
            next_obs, rews_env, terminated, truncated, info = env.step(acts)
            done = terminated or truncated

            r_mission  = info['r_mission']
            r_safety   = info['r_safety']
            pah_inputs = info['pah_inputs']

            buffer.add(obs, acts, val_m, val_s, lps, done, r_mission, r_safety, pah_inputs)
            obs = next_obs
            if done:
                ep_count += 1
                obs = env.reset()

        stats = agent.update(buffer, obs)
        a_loss, v_loss, entropy = stats['actor_loss'], stats['critic_loss'], stats['entropy']
        buffer.clear()

        if ep_count >= last_eval + CFG['eval_every']:
            successes, collisions = 0, 0
            eval_alphas, eval_taus = [], []
            for _ in range(CFG['eval_episodes']):
                o    = eval_env.reset()
                done = False
                while not done:
                    a, _, _, _ = agent.get_actions(o)
                    o, _, terminated, truncated, info = eval_env.step(a)
                    done = terminated or truncated
                    pi = info['pah_inputs']
                    al = agent.compute_alpha(pi['tau'], pi['d_target'], pi['n_conflict'])
                    eval_alphas.append(al)
                    eval_taus.append(pi['tau'])
                if info['all_targets_reached'] and not info['episode_collision']:
                    successes += 1
                if info['episode_collision']:
                    collisions += 1

            sr      = successes  / CFG['eval_episodes']
            cr      = collisions / CFG['eval_episodes']
            elapsed = (time.time() - start) / 60
            log_std = agent.actor.log_std.data.cpu().numpy()

            eval_alphas  = np.concatenate(eval_alphas)
            eval_taus    = np.concatenate(eval_taus)
            danger_mask  = eval_taus <= (CFG['pah_horizon'] * 0.34)
            safe_mask    = eval_taus >= (CFG['pah_horizon'] * 0.9)
            alpha_low_tau  = float(eval_alphas[danger_mask].mean()) if danger_mask.any() else float('nan')
            alpha_high_tau = float(eval_alphas[safe_mask].mean())   if safe_mask.any()   else float('nan')

            print(" {:>9} | {:>7.1%} | {:>9.1%} | {:>+8.4f} | {:>8.4f} | {:>10.3f} | {:>6.1f}m".format(
                ep_count, sr, cr, a_loss, entropy, stats['alpha_mean'], elapsed
            ))
            last_eval = ep_count

            history['episode'].append(ep_count)
            history['success'].append(sr)
            history['collision'].append(cr)
            history['actor_loss'].append(a_loss)
            history['critic_loss'].append(v_loss)
            history['entropy'].append(entropy)
            history['log_std'].append(log_std.tolist())
            history['alpha_mean'].append(stats['alpha_mean'])
            history['alpha_std'].append(stats['alpha_std'])
            history['pah_loss'].append(stats.get('pah_loss'))
            history['eval_alpha_low_tau'].append(alpha_low_tau)
            history['eval_alpha_high_tau'].append(alpha_high_tau)

        if ep_count >= last_save + CFG['save_every']:
            agent.save("{}/checkpoint_ep{}.pt".format(RESULTS_DIR, ep_count))
            last_save = ep_count
            print("  >> checkpoint saved: ep{}".format(ep_count))

    agent.save('{}/final_model.pt'.format(RESULTS_DIR))
    with open('{}/history.json'.format(RESULTS_DIR), 'w') as f:
        json_module.dump(history, f, indent=2)

    total_min = (time.time() - start) / 60
    print('\nSeed {} done in {:.1f}m. Saved to {}'.format(seed, total_min, RESULTS_DIR))

    if CFG['use_pah']:
        _hi, _lo = history['eval_alpha_high_tau'][-1], history['eval_alpha_low_tau'][-1]
        if not (np.isnan(_hi) or np.isnan(_lo)):
            print('Final alpha(safe) - alpha(danger) = {:+.3f}'.format(_hi - _lo))

    plot_results(history, RESULTS_DIR, seed)
    return history


all_histories = {}
for _seed in SEEDS:
    all_histories[_seed] = run_training(_seed)

print("\n" + "=" * 80)
print("ALL {} SEEDS DONE: {}".format(len(SEEDS), SEEDS))
print("=" * 80)


In [ ]:
# Cell 7 — Combined summary across all 5 seeds

def summarize(history):
    n = len(history['episode'])
    avg_s = sum(history['success']) / n * 100
    avg_c = sum(history['collision']) / n * 100
    third = max(n // 3, 1)
    ft_s  = sum(history['success'][:third]) / third * 100
    lt_s  = sum(history['success'][n-third:]) / third * 100
    ent0, ent1 = history['entropy'][0], history['entropy'][-1]

    right = wrong = 0
    ht_map = dict(zip(history['episode'], history['eval_alpha_high_tau']))
    for e, lv in zip(history['episode'], history['eval_alpha_low_tau']):
        if lv != lv:  # nan
            continue
        hv = ht_map.get(e, float('nan'))
        if hv != hv:
            continue
        if lv >= hv:
            wrong += 1
        else:
            right += 1

    return dict(avg_s=avg_s, avg_c=avg_c, ft_s=ft_s, lt_s=lt_s,
                ent0=ent0, ent1=ent1, right=right, wrong=wrong)


print(" {:>6} | {:>8} | {:>10} | {:>14} | {:>14} | {:>14}".format(
    "Seed", "Avg Succ", "Avg Coll", "First->Last %", "Entropy end", "Direction (r/w)"
))
print('-' * 90)

tot_right = tot_wrong = 0
sum_s = sum_c = 0.0
for _seed in SEEDS:
    stats = summarize(all_histories[_seed])
    tot_right += stats['right']
    tot_wrong += stats['wrong']
    sum_s += stats['avg_s']
    sum_c += stats['avg_c']
    dir_str = '{}/{}'.format(stats['right'], stats['right'] + stats['wrong']) if CFG['use_pah'] else 'n/a (fixed)'
    print(" {:>6} | {:>7.1f}% | {:>9.1f}% | {:>6.1f}->{:<6.1f} | {:>14.3f} | {:>14}".format(
        _seed, stats['avg_s'], stats['avg_c'], stats['ft_s'], stats['lt_s'], stats['ent1'], dir_str
    ))

print('-' * 90)
print(" {:>6} | {:>7.1f}% | {:>9.1f}% |".format('AVG', sum_s / len(SEEDS), sum_c / len(SEEDS)))

if CFG['use_pah'] and (tot_right + tot_wrong) > 0:
    pct_wrong = 100 * tot_wrong / (tot_right + tot_wrong)
    print("\nCombined direction check across all {} seeds: {}/{} right, {}/{} wrong ({:.0f}%)".format(
        len(SEEDS), tot_right, tot_right + tot_wrong, tot_wrong, tot_right + tot_wrong, pct_wrong
    ))
    print("Compare to stage2-pah-v0 (single shared critic): 27/79 wrong (34%) across the same 5 seeds.")
    print("If this two-head critic run is meaningfully below 34%, the escalation worked.")

import json as json_module
with open('combined_summary.json', 'w') as f:
    json_module.dump({
        str(_seed): summarize(all_histories[_seed]) for _seed in SEEDS
    }, f, indent=2)
print("\nSaved combined_summary.json")
